In [26]:
import pandas as pd
import os

In [30]:
instrument_ids = [d for d in os.listdir("../data/ticks") if os.path.isdir(os.path.join("../data/ticks", d))]

In [2]:
date = "2026-05-13"

In [35]:
def get_file_path(instrument_id , date):
    return f"../data/ticks/{instrument_id}/{date}.jsonl"

In [39]:
thresholds = {}

for instrument_id in instrument_ids:
    df = pd.read_json(get_file_path(instrument_id, date), lines=True)

    df['ts_parsed'] = pd.to_datetime(df['ts'])
    df['ts_diff'] = df['ts_parsed'].diff()
    df['ts_diff_seconds'] = df['ts_diff'].dt.total_seconds()

    thresholds[instrument_id] = {
        'p99':  df['ts_diff_seconds'].quantile(0.99),
        'mean': df['ts_diff_seconds'].mean(),
    }

In [40]:
thresholds

{'IRTKZFAM0001': {'p99': np.float64(16.489565839999905),
  'mean': np.float64(3.1791302998312845)},
 'IRTKJAVA0001': {'p99': np.float64(16.205439780000088),
  'mean': np.float64(3.415253960652343)},
 'IRTKGANJ0001': {'p99': np.float64(6.28755422),
  'mean': np.float64(2.1483724836399154)},
 'IRTKTABA0001': {'p99': np.float64(26.303650360000002),
  'mean': np.float64(3.9992950812860175)},
 'IRTKZOMR0001': {'p99': np.float64(20.3765985),
  'mean': np.float64(3.6901829264541384)},
 'IRTKROSE0001': {'p99': np.float64(17.61720165999999),
  'mean': np.float64(3.0492419861335796)},
 'IRTKKIAN0001': {'p99': np.float64(8.861031849999893),
  'mean': np.float64(2.5186663020992364)},
 'IRTKALTN0001': {'p99': np.float64(14.062564000000005),
  'mean': np.float64(3.007673682926829)},
 'IRTKATSH0001': {'p99': np.float64(22.21074008),
  'mean': np.float64(3.2375002347740667)},
 'IRTKDRKS0001': {'p99': np.float64(13.591190879999981),
  'mean': np.float64(2.961057768454117)},
 'IRTKLOTF0001': {'p99': np.

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10835 entries, 0 to 10834
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   isin       10835 non-null  str   
 1   ts         10835 non-null  str   
 2   stream_id  10835 non-null  str   
 3   snapshot   10835 non-null  object
dtypes: object(1), str(3)
memory usage: 963.0+ KB


In [7]:
df.head()

,isin,ts,stream_id,snapshot
0,IRTKMOFD0001,2026-05-13T09:50:05.250631+00:00,1778665805245-0,"{'time': '09:50:05', 'depths': [{'depth': 1, '..."
1,IRTKMOFD0001,2026-05-13T09:50:07.317222+00:00,1778665807312-0,"{'time': '09:50:07', 'depths': [{'depth': 1, '..."
2,IRTKMOFD0001,2026-05-13T09:50:07.319849+00:00,1778665807314-0,"{'time': '09:50:07', 'depths': [{'depth': 1, '..."
3,IRTKMOFD0001,2026-05-13T09:50:08.025194+00:00,1778665808020-0,"{'time': '09:50:08', 'depths': [{'depth': 1, '..."
4,IRTKMOFD0001,2026-05-13T09:50:08.862387+00:00,1778665808857-0,"{'time': '09:50:08', 'depths': [{'depth': 1, '..."


In [9]:
df['ts_parsed'] = pd.to_datetime(df['ts'])
df['ts_diff'] = df['ts_parsed'].diff()
df['ts_diff_seconds'] = df['ts_diff'].dt.total_seconds()

# remove the max outlier (manual disconnection)
max_gap_idx = df['ts_diff_seconds'].idxmax()
df_clean = df.drop(index=max_gap_idx).reset_index(drop=True)

print(f"removed row {max_gap_idx} with gap: {df['ts_diff_seconds'].max():.1f}s")
print(f"\nafter removal:")
print(f"min gap:  {df_clean['ts_diff_seconds'].min():.3f}s")
print(f"max gap:  {df_clean['ts_diff_seconds'].max():.3f}s")
print(f"mean gap: {df_clean['ts_diff_seconds'].mean():.3f}s")

removed row 1570 with gap: 2616.6s

after removal:
min gap:  0.001s
max gap:  30.596s
mean gap: 0.977s


In [24]:
threshold = df_clean['ts_diff_seconds'].quantile(0.99)
print(f"99th percentile threshold: {threshold:.3f}s")

99th percentile threshold: 2.258s
